# GNN Experiment: Graph-Conditioned Configuration Ranker

This notebook is a focused experiment for testing whether a memory-safe GNN can improve validation ranking on the layout XLA collections.

It does **not** generate `submission.csv`. The goal is to answer one question first:

```text
Can a k-hop configurable-node GraphSAGE ranker beat the current tree models on layout:xla validation?
```


## 1. Download Kaggle Data in Colab

This block is copied from the executed Colab workflow. It expects `KAGGLE_USERNAME` and `KAGGLE_KEY` to exist in Colab Secrets.


In [ ]:
from google.colab import userdata

username = userdata.get("KAGGLE_USERNAME")
key = userdata.get("KAGGLE_KEY")

print("Username exists:", username is not None)
print("Key exists:", key is not None)
from google.colab import userdata
import os

import kagglehub

# Get Kaggle credentials from Colab Secrets.
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

# Check that credentials were loaded.
print("Username loaded:", bool(os.environ.get("KAGGLE_USERNAME")))
print("Key loaded:", bool(os.environ.get("KAGGLE_KEY")))

# Download directly into /content so it is visible in Colab Files.
path = kagglehub.competition_download(
    "predict-ai-model-runtime",
    output_dir="/content/predict-ai-model-runtime"
)

print("Downloaded to:", path)

# Keep this output short. Detailed file counts are shown in the inspection section.
print("\nTop-level files/folders:")
for item in sorted(os.listdir(path)):
    print("-", item)


## 2. Imports and Configuration

The GNN uses plain PyTorch rather than PyTorch Geometric. This keeps installation simple and avoids heavy graph-library dependencies.


In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path
import os
import time
import gc
import warnings


def ensure_package(package_name, import_name=None):
    import_name = import_name or package_name
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])


ensure_package("numpy")
ensure_package("pandas")
ensure_package("scikit-learn", "sklearn")
ensure_package("tqdm")

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error
from tqdm.auto import tqdm

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    TORCH_AVAILABLE = True
except Exception as exc:
    TORCH_AVAILABLE = False
    raise RuntimeError("PyTorch is required for this GNN experiment notebook") from exc

warnings.filterwarnings("ignore")
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch:", torch.__version__)
print("Device:", DEVICE)

# Keep these conservative for Colab RAM. Increase only after a successful smoke test.
GNN_COLLECTIONS = ["layout:xla:default", "layout:xla:random"]
GNN_SUBGRAPH_HOPS = 2
GNN_MAX_SUBGRAPH_NODES = 512
GNN_MAX_TRAIN_FILES = 8
GNN_MAX_VALID_FILES = 5
GNN_MAX_TRAIN_CONFIGS_PER_FILE = 64
GNN_MAX_VALID_CONFIGS_PER_FILE = 1000
GNN_BATCH_SIZE = 4
GNN_HIDDEN_DIM = 32
GNN_EPOCHS = 3
GNN_LR = 1e-3
PAIR_MARGIN = 0.1
PAIR_MIN_LOG_RUNTIME_GAP = 0.02
PAIR_MAX_PAIRS = 512

print("GNN_COLLECTIONS:", GNN_COLLECTIONS)
print("GNN_SUBGRAPH_HOPS:", GNN_SUBGRAPH_HOPS)
print("GNN_MAX_SUBGRAPH_NODES:", GNN_MAX_SUBGRAPH_NODES)
print("GNN_BATCH_SIZE:", GNN_BATCH_SIZE)


## 3. Locate Data

Only the two layout XLA collections are used here. The other collections stay in the main notebook.


In [ ]:
def find_data_root():
    candidates = [
        Path.cwd() / "data",
        Path.cwd().parent / "data",
        Path("/content/data"),
        Path.cwd() / "predict-ai-model-runtime",
        Path.cwd().parent / "predict-ai-model-runtime",
        Path("/content/predict-ai-model-runtime"),
        Path.cwd(),
    ]
    for candidate in candidates:
        if (candidate / "npz_all" / "npz").exists():
            return candidate
    raise FileNotFoundError("Could not find npz_all/npz. Put the Kaggle data folder in data/ or update find_data_root().")


DATA_ROOT = find_data_root()
NPZ_ROOT = DATA_ROOT / "npz_all" / "npz"
COLLECTIONS = {
    "layout:xla:default": NPZ_ROOT / "layout" / "xla" / "default",
    "layout:xla:random": NPZ_ROOT / "layout" / "xla" / "random",
}

print("DATA_ROOT:", DATA_ROOT)
for name, path in COLLECTIONS.items():
    print(name, "train", len(list((path / "train").glob("*.npz"))), "valid", len(list((path / "valid").glob("*.npz"))))


def split_files(collection_name, split):
    return sorted((COLLECTIONS[collection_name] / split).glob("*.npz"))


## 4. Sampling and Validation Utilities

In [ ]:
def get_num_configs(data):
    if "node_config_feat" in data:
        return data["node_config_feat"].shape[0]
    if "config_feat" in data:
        return data["config_feat"].shape[0]
    raise KeyError("Could not find config features")


def choose_indices(n_items, max_items=None, seed=RANDOM_SEED):
    if max_items is None or n_items <= max_items:
        return np.arange(n_items, dtype=np.int64)
    local_rng = np.random.default_rng(seed)
    return np.sort(local_rng.choice(n_items, size=max_items, replace=False)).astype(np.int64)


def choose_runtime_stratified_indices(runtimes, max_items, seed=RANDOM_SEED):
    runtimes = np.asarray(runtimes, dtype=np.float64)
    valid_idx = np.flatnonzero(np.isfinite(runtimes) & (runtimes > 0))
    if max_items is None or len(valid_idx) <= max_items:
        return np.arange(len(runtimes), dtype=np.int64)
    if len(valid_idx) == 0:
        return choose_indices(len(runtimes), max_items=max_items, seed=seed)

    local_rng = np.random.default_rng(seed)
    sorted_idx = valid_idx[np.argsort(runtimes[valid_idx])]
    fastest_count = max(1, int(max_items * 0.20))
    slowest_count = max(1, int(max_items * 0.10))
    selected = set(sorted_idx[:fastest_count].tolist())
    selected.update(sorted_idx[-slowest_count:].tolist())

    remaining = np.array([idx for idx in valid_idx if idx not in selected], dtype=np.int64)
    budget = max_items - len(selected)
    if budget > 0 and len(remaining) > 0:
        chosen = local_rng.choice(remaining, size=min(budget, len(remaining)), replace=False)
        selected.update(chosen.tolist())
    return np.array(sorted(selected), dtype=np.int64)


def choose_config_indices(data, split, max_items=None, seed=RANDOM_SEED):
    n_items = get_num_configs(data)
    if max_items is None or n_items <= max_items:
        return np.arange(n_items, dtype=np.int64)
    if split in ["train", "valid"] and "config_runtime" in data:
        return choose_runtime_stratified_indices(data["config_runtime"], max_items=max_items, seed=seed)
    return choose_indices(n_items, max_items=max_items, seed=seed)


def sampled_kendall_score(y_true, y_pred, max_pairs=20000, seed=RANDOM_SEED):
    n = len(y_true)
    if n < 2:
        return np.nan
    local_rng = np.random.default_rng(seed)
    i = local_rng.integers(0, n, size=max_pairs)
    j = local_rng.integers(0, n, size=max_pairs)
    mask = i != j
    i, j = i[mask], j[mask]
    true_order = np.sign(y_true[i] - y_true[j])
    pred_order = np.sign(y_pred[i] - y_pred[j])
    useful = (true_order != 0) & (pred_order != 0)
    if useful.sum() == 0:
        return np.nan
    return float(np.mean(true_order[useful] == pred_order[useful]) * 2 - 1)


def centered_log_runtime(runtimes):
    log_runtime = np.log1p(np.asarray(runtimes, dtype=np.float64))
    return log_runtime - np.median(log_runtime)


## 5. K-Hop Subgraph Around Configurable Nodes

The subgraph starts from all `node_config_ids`, expands by `GNN_SUBGRAPH_HOPS`, and keeps one combined induced subgraph per graph file.


In [ ]:
def build_adjacency(edge_index, node_count):
    neighbors = [set() for _ in range(node_count)]
    for src, dst in np.asarray(edge_index, dtype=np.int64):
        if 0 <= src < node_count and 0 <= dst < node_count:
            neighbors[int(src)].add(int(dst))
            neighbors[int(dst)].add(int(src))
    return neighbors


def khop_subgraph_nodes(edge_index, node_count, seed_nodes, hops=2, max_nodes=512):
    neighbors = build_adjacency(edge_index, node_count)
    seed_nodes = [int(n) for n in seed_nodes if 0 <= int(n) < node_count]
    if not seed_nodes:
        return np.arange(min(node_count, max_nodes), dtype=np.int64)

    reached = set(seed_nodes)
    frontier = set(seed_nodes)
    ordered = list(dict.fromkeys(seed_nodes))

    for _ in range(hops):
        next_frontier = set()
        for node in sorted(frontier):
            for nbr in sorted(neighbors[node]):
                if nbr not in reached:
                    reached.add(nbr)
                    next_frontier.add(nbr)
                    ordered.append(nbr)
        frontier = next_frontier
        if not frontier:
            break

    # Always keep configurable nodes first, then nearest discovered neighbours.
    if len(ordered) > max_nodes:
        seed_set = list(dict.fromkeys(seed_nodes))
        remaining = [node for node in ordered if node not in set(seed_set)]
        ordered = seed_set + remaining[:max(0, max_nodes - len(seed_set))]
    return np.array(sorted(ordered), dtype=np.int64)


def induced_edges(edge_index, kept_nodes):
    kept_nodes = np.asarray(kept_nodes, dtype=np.int64)
    old_to_new = {int(old): i for i, old in enumerate(kept_nodes)}
    src_list = []
    dst_list = []
    for src, dst in np.asarray(edge_index, dtype=np.int64):
        if int(src) in old_to_new and int(dst) in old_to_new:
            src_list.append(old_to_new[int(src)])
            dst_list.append(old_to_new[int(dst)])
            src_list.append(old_to_new[int(dst)])
            dst_list.append(old_to_new[int(src)])
    for i in range(len(kept_nodes)):
        src_list.append(i)
        dst_list.append(i)
    return np.asarray(src_list, dtype=np.int64), np.asarray(dst_list, dtype=np.int64), old_to_new


def inspect_subgraph_sizes(max_files=5):
    rows = []
    for collection_name in GNN_COLLECTIONS:
        for file_path in split_files(collection_name, "train")[:max_files]:
            with np.load(file_path) as data:
                node_count = int(data["node_feat"].shape[0])
                node_config_ids = np.asarray(data["node_config_ids"], dtype=np.int64)
                kept = khop_subgraph_nodes(data["edge_index"], node_count, node_config_ids, hops=GNN_SUBGRAPH_HOPS, max_nodes=GNN_MAX_SUBGRAPH_NODES)
                rows.append({
                    "collection": collection_name,
                    "file": file_path.stem,
                    "node_count": node_count,
                    "configurable_nodes": len(node_config_ids),
                    "subgraph_nodes": len(kept),
                })
    return pd.DataFrame(rows)


display(inspect_subgraph_sizes())


## 6. GraphSAGE Ranker

The model outputs one scalar score per configuration. Lower score means faster predicted runtime.


In [ ]:
class GraphSAGERanker(nn.Module):
    def __init__(self, node_dim, config_dim, hidden_dim=GNN_HIDDEN_DIM, opcode_vocab_size=256, opcode_emb_dim=16):
        super().__init__()
        self.opcode_embedding = nn.Embedding(opcode_vocab_size, opcode_emb_dim)
        self.input_proj = nn.Linear(node_dim + config_dim + opcode_emb_dim, hidden_dim)
        self.sage1 = nn.Linear(hidden_dim * 2, hidden_dim)
        self.sage2 = nn.Linear(hidden_dim * 2, hidden_dim)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 4, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.05),
            nn.Linear(hidden_dim, 1),
        )

    def sage_step(self, h, edge_src, edge_dst, degree, layer):
        neigh = torch.zeros_like(h)
        neigh.index_add_(1, edge_dst, h[:, edge_src, :])
        neigh = neigh / degree.view(1, -1, 1).clamp_min(1.0)
        return F.relu(layer(torch.cat([h, neigh], dim=-1))) + h

    def forward(self, base_node, opcode, config_node, edge_src, edge_dst, degree, config_mask):
        batch_size = config_node.shape[0]
        base = base_node.unsqueeze(0).expand(batch_size, -1, -1)
        opcode_emb = self.opcode_embedding(opcode).unsqueeze(0).expand(batch_size, -1, -1)
        h = F.relu(self.input_proj(torch.cat([base, config_node, opcode_emb], dim=-1)))
        h = self.sage_step(h, edge_src, edge_dst, degree, self.sage1)
        h = self.sage_step(h, edge_src, edge_dst, degree, self.sage2)

        global_pool = torch.cat([h.mean(dim=1), h.max(dim=1).values], dim=-1)
        mask = config_mask.view(1, -1, 1).float()
        denom = mask.sum(dim=1).clamp_min(1.0)
        config_mean = (h * mask).sum(dim=1) / denom
        config_max = h.masked_fill(mask == 0, -1e9).max(dim=1).values
        config_max = torch.where(torch.isfinite(config_max), config_max, torch.zeros_like(config_max))
        config_pool = torch.cat([config_mean, config_max], dim=-1)
        return self.head(torch.cat([global_pool, config_pool], dim=-1)).squeeze(-1)


def pairwise_margin_ranking_loss(scores, runtimes, margin=PAIR_MARGIN, min_gap=PAIR_MIN_LOG_RUNTIME_GAP, max_pairs=PAIR_MAX_PAIRS):
    log_runtime = torch.log1p(runtimes)
    diff = log_runtime.view(-1, 1) - log_runtime.view(1, -1)
    pairs = torch.nonzero(diff < -min_gap, as_tuple=False)
    if pairs.numel() == 0:
        return F.smooth_l1_loss(scores, log_runtime - log_runtime.median())
    if len(pairs) > max_pairs:
        idx = torch.randperm(len(pairs), device=scores.device)[:max_pairs]
        pairs = pairs[idx]
    fast = pairs[:, 0]
    slow = pairs[:, 1]
    return F.relu(margin + scores[fast] - scores[slow]).mean()


## 7. Model Wrapper

In [ ]:
class LayoutGNNRanker:
    def __init__(self):
        self.device = DEVICE
        self.model = None
        self.node_dim = None
        self.config_dim = None
        self.history = []

    def base_node_features(self, data, kept_nodes):
        node_feat = np.asarray(data["node_feat"][kept_nodes], dtype=np.float32)
        node_feat = np.log1p(np.maximum(node_feat, 0.0))
        return (node_feat - node_feat.mean(axis=0, keepdims=True)) / (node_feat.std(axis=0, keepdims=True) + 1e-6)

    def prepare_graph(self, data):
        node_count = int(data["node_feat"].shape[0])
        raw_config_nodes = np.asarray(data["node_config_ids"], dtype=np.int64)
        kept_nodes = khop_subgraph_nodes(data["edge_index"], node_count, raw_config_nodes, hops=GNN_SUBGRAPH_HOPS, max_nodes=GNN_MAX_SUBGRAPH_NODES)
        edge_src_np, edge_dst_np, old_to_new = induced_edges(data["edge_index"], kept_nodes)

        config_local_pairs = [(pos, old_to_new[int(old)]) for pos, old in enumerate(raw_config_nodes) if int(old) in old_to_new]
        config_positions = np.array([p for p, _ in config_local_pairs], dtype=np.int64)
        config_local_nodes = np.array([n for _, n in config_local_pairs], dtype=np.int64)
        config_mask = np.zeros(len(kept_nodes), dtype=np.float32)
        config_mask[config_local_nodes] = 1.0

        degree = np.bincount(edge_dst_np, minlength=len(kept_nodes)).astype(np.float32)
        base_node = torch.tensor(self.base_node_features(data, kept_nodes), dtype=torch.float32, device=self.device)
        opcode = np.clip(np.asarray(data["node_opcode"][kept_nodes], dtype=np.int64), 0, 255)
        opcode = torch.tensor(opcode, dtype=torch.long, device=self.device)
        edge_src = torch.tensor(edge_src_np, dtype=torch.long, device=self.device)
        edge_dst = torch.tensor(edge_dst_np, dtype=torch.long, device=self.device)
        degree = torch.tensor(degree, dtype=torch.float32, device=self.device)
        config_mask = torch.tensor(config_mask, dtype=torch.float32, device=self.device)
        return base_node, opcode, edge_src, edge_dst, degree, config_mask, config_positions, config_local_nodes

    def ensure_model(self, data):
        node_dim = int(data["node_feat"].shape[1])
        config_dim = int(data["node_config_feat"].shape[2])
        if self.model is None:
            self.node_dim = node_dim
            self.config_dim = config_dim
            self.model = GraphSAGERanker(node_dim=node_dim, config_dim=config_dim).to(self.device)
        return self.model

    def config_tensor(self, data, config_indices, n_nodes, config_positions, config_local_nodes):
        selected = np.asarray(data["node_config_feat"][config_indices], dtype=np.float32)
        selected = np.where(selected == -1, 0.0, selected)
        config_node = np.zeros((len(config_indices), n_nodes, selected.shape[2]), dtype=np.float32)
        if len(config_positions):
            config_node[:, config_local_nodes, :] = selected[:, config_positions, :]
        return torch.tensor(config_node, dtype=torch.float32, device=self.device)

    def fit(self, collection_name, files):
        first_file = files[0]
        with np.load(first_file) as data:
            self.ensure_model(data)
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=GNN_LR, weight_decay=1e-4)

        for epoch in range(1, GNN_EPOCHS + 1):
            losses = []
            progress = tqdm(files, desc=f"{collection_name} epoch {epoch}/{GNN_EPOCHS}", unit="file")
            for file_id, file_path in enumerate(progress):
                with np.load(file_path) as data:
                    self.ensure_model(data)
                    config_indices = choose_config_indices(data, "train", max_items=GNN_MAX_TRAIN_CONFIGS_PER_FILE, seed=RANDOM_SEED + file_id)
                    runtimes = np.asarray(data["config_runtime"], dtype=np.float32)[config_indices]
                    graph_tensors = self.prepare_graph(data)
                    base_node, opcode, edge_src, edge_dst, degree, config_mask, config_positions, config_local_nodes = graph_tensors
                    order = np.arange(len(config_indices))
                    np.random.default_rng(RANDOM_SEED + epoch + file_id).shuffle(order)

                    for start in range(0, len(order), GNN_BATCH_SIZE):
                        batch_pos = order[start:start + GNN_BATCH_SIZE]
                        batch_config_indices = config_indices[batch_pos]
                        batch_runtime = torch.tensor(runtimes[batch_pos], dtype=torch.float32, device=self.device)
                        config_node = self.config_tensor(data, batch_config_indices, len(base_node), config_positions, config_local_nodes)
                        scores = self.model(base_node, opcode, config_node, edge_src, edge_dst, degree, config_mask)
                        loss = pairwise_margin_ranking_loss(scores, batch_runtime)
                        optimizer.zero_grad()
                        loss.backward()
                        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                        optimizer.step()
                        losses.append(float(loss.detach().cpu()))
                    progress.set_postfix(loss=np.mean(losses[-10:]) if losses else np.nan)
                    del graph_tensors
            mean_loss = float(np.mean(losses)) if losses else np.nan
            self.history.append({"epoch": epoch, "loss": mean_loss})
            print(f"{collection_name} epoch {epoch}: mean_loss={mean_loss:.5f}")
            gc.collect()
        return self

    def predict_file(self, file_path, split="valid", max_configs=None, seed=RANDOM_SEED):
        self.model.eval()
        with np.load(file_path) as data:
            config_indices = choose_config_indices(data, split, max_items=max_configs, seed=seed)
            graph_tensors = self.prepare_graph(data)
            base_node, opcode, edge_src, edge_dst, degree, config_mask, config_positions, config_local_nodes = graph_tensors
            preds = []
            with torch.no_grad():
                for start in range(0, len(config_indices), GNN_BATCH_SIZE):
                    batch_indices = config_indices[start:start + GNN_BATCH_SIZE]
                    config_node = self.config_tensor(data, batch_indices, len(base_node), config_positions, config_local_nodes)
                    score = self.model(base_node, opcode, config_node, edge_src, edge_dst, degree, config_mask)
                    preds.append(score.detach().cpu().numpy())
        return config_indices, np.concatenate(preds) if preds else np.array([], dtype=np.float64)


## 8. Train and Validate Layout GNNs

This trains one model per layout XLA collection and reports sampled Kendall ranking score. Higher is better.


In [ ]:
def validate_model(model, collection_name):
    files = split_files(collection_name, "valid")[:GNN_MAX_VALID_FILES]
    rows = []
    for i, file_path in enumerate(tqdm(files, desc=f"{collection_name} validation", unit="file")):
        with np.load(file_path) as data:
            config_indices = choose_config_indices(data, "valid", max_items=GNN_MAX_VALID_CONFIGS_PER_FILE, seed=RANDOM_SEED + i)
            y_true = np.asarray(data["config_runtime"], dtype=np.float64)[config_indices]
        _, pred = model.predict_file(file_path, split="valid", max_configs=GNN_MAX_VALID_CONFIGS_PER_FILE, seed=RANDOM_SEED + i)
        rows.append({
            "collection": collection_name,
            "file": file_path.stem,
            "n_configs": len(y_true),
            "ranking_score": sampled_kendall_score(y_true, pred, seed=RANDOM_SEED + i),
            "centered_log_mae": mean_absolute_error(centered_log_runtime(y_true), pred),
        })
    return pd.DataFrame(rows)


models = {}
validation_tables = []

for collection_name in GNN_COLLECTIONS:
    train_files = split_files(collection_name, "train")[:GNN_MAX_TRAIN_FILES]
    print("\n" + "=" * 80)
    print("Training GNN:", collection_name)
    print("train files:", len(train_files))
    start = time.perf_counter()
    model = LayoutGNNRanker().fit(collection_name, train_files)
    models[collection_name] = model
    valid_df = validate_model(model, collection_name)
    valid_df["train_seconds"] = round(time.perf_counter() - start, 2)
    validation_tables.append(valid_df)
    display(valid_df)

validation_df = pd.concat(validation_tables, ignore_index=True)
summary = validation_df.groupby("collection", as_index=False).agg(
    valid_files=("file", "count"),
    ranking_score=("ranking_score", "mean"),
    centered_log_mae=("centered_log_mae", "mean"),
    train_seconds=("train_seconds", "max"),
)
display(summary)


## 9. How To Interpret The Result

Compare the `ranking_score` above against the tree-model validation scores from the main notebook. If the GNN is clearly worse, stop here. If it is competitive on `layout:xla:*`, then integrate it into `SC4000_Eugene.ipynb` for final submission generation.

This experiment intentionally does not write `submission.csv`.
